<pre>
- Partículas com diâmetro inferior a 2,5 µm (PM2,5) 

Unidade de medida de retorno é kg/m³ (quilograma por metro cúbico)

</pre>


In [1]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [2]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
dataset = "cams-global-reanalysis-eac4"
request = {
    "variable": [
        "particulate_matter_2.5um"
    ],
    "date": ["2025-12-01/2025-12-31"],
    "time": ["06:00"],
    "data_format": "netcdf",
    "area": [6, -74, -35, -34]
}


client = cdsapi.Client(
    url = "https://ads.atmosphere.copernicus.eu/api",
    key = "34161618-bf6b-41ca-9272-50b917f789b9"
)

ret_download = client.retrieve(dataset, request).download()

print(f"Download completed: {ret_download}")

2026-07-22 15:59:09,014 INFO Request ID is 71f0e7d4-21b2-434c-a32e-15452e59b394
2026-07-22 15:59:09,213 INFO status has been updated to accepted
2026-07-22 15:59:23,875 INFO status has been updated to running
2026-07-22 15:59:31,754 INFO status has been updated to successful
                                                                                      

Download completed: b3369884ba6b697f7c272669357c077a.nc


In [4]:
with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Poluicao\\{ret_download}"
                    ,engine="netcdf4"
                    ,chunks={"time": 365
                            ,"latitude": 100
                            ,"longitude": 100 }
                    ) as ds:
    # Transforma o Dataset em um Spark Dataframe
    df_dask       = ds.to_dask_dataframe()
    df_dask_c     = df_dask.compute()
    df_particulas = spark.createDataFrame(df_dask_c)

df_particulas.printSchema()
df_particulas.show(10, False)

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- valid_time: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- pm2p5: float (nullable = true)

+-------------------+--------+---------+-------------+
|valid_time         |latitude|longitude|pm2p5        |
+-------------------+--------+---------+-------------+
|2025-12-01 06:00:00|6.0     |-73.5    |1.3518502E-8 |
|2025-12-01 06:00:00|6.0     |-72.75   |1.1508746E-8 |
|2025-12-01 06:00:00|6.0     |-72.0    |9.887799E-9  |
|2025-12-01 06:00:00|6.0     |-71.25   |1.0358235E-8 |
|2025-12-01 06:00:00|6.0     |-70.5    |8.442953E-9  |
|2025-12-01 06:00:00|6.0     |-69.75   |1.40914835E-8|
|2025-12-01 06:00:00|6.0     |-69.0    |1.5902629E-8 |
|2025-12-01 06:00:00|6.0     |-68.25   |1.0136546E-8 |
|2025-12-01 06:00:00|6.0     |-67.5    |7.665335E-9  |
|2025-12-01 06:00:00|6.0     |-66.75   |6.428877E-9  |
+-------------------+--------+---------+-------------+
only showing top 10 rows


In [ ]:
# Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)

# Fator de conversão: 1 kg = 1e9 micrograms
FATOR_CONVERSAO_PM25 = 1e9 # 1000000000

drop_cols = ["valid_time", "pm2p5"]

df_particulas_pm2p5 = \
    (df_particulas
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("Poluição do ar - PM₂.₅ (µg/m3)") 
                     ,"valor": (F.col("pm2p5") * F.lit(FATOR_CONVERSAO_PM25)).cast("double")
                     ,"unidade_medida": F.lit("µg/m3")})
         .drop(*drop_cols)

    )

In [6]:
# df_particulas_pm2p5.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_particulas_pm2p5.csv", index=False)

df_particulas_pm2p5.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_particulas_pm2p5.parquet")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [7]:
df_particulas_parquet = \
    spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_particulas_pm2p5.parquet")

df_particulas_parquet.printSchema()
df_particulas_parquet.show(10, False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+--------+---------+------------+------------------------------+------------------+--------------+
|latitude|longitude|data_medicao|indicador                     |valor             |unidade_medida|
+--------+---------+------------+------------------------------+------------------+--------------+
|6.0     |-73.5    |2025-12-01  |Poluição do ar - PM₂.₅ (µg/m3)|13.518501873477362|µg/m3         |
|6.0     |-72.75   |2025-12-01  |Poluição do ar - PM₂.₅ (µg/m3)|11.508745956234634|µg/m3         |
|6.0     |-72.0    |2025-12-01  |Poluição do ar - PM₂.₅ (µg/m3)|9.887799023999833 |µg/m3         |
|6.0     |-71.25   |2025-12-01  |Poluição do ar - PM₂.₅ (µg/m3)|10.35823515849188 |µg/m3         |
|6.0     |-70.5    |2025-12-01  |Poluição do ar - PM₂.

In [8]:
os.remove(r"C:\Marco Conti\Projetos\MAIS-v2\Poluicao\{file_name}".format(file_name = ret_download))